# Clasificación supervisada
## Comparación de representaciones tiempo-frecuencia: STFT vs CWT

Este notebook implementa la etapa final del proyecto: entrenar y evaluar
modelos de clasificación supervisada sobre los datasets de features extraídos
en los notebooks anteriores, con el objetivo de determinar qué representación
tiempo-frecuencia produce features de mayor utilidad predictiva.

---

## Contexto: lo que sabemos antes de clasificar

El análisis estadístico del notebook 05 estableció los siguientes hallazgos,
que informan directamente las decisiones de diseño de este notebook:

1. **La CWT supera a la STFT en las cuatro métricas de discriminabilidad**
   (Cohen's $d$, Mann-Whitney, FDR y Mutual Information), con una ventaja
   aproximada del doble en FDR y MI.

2. **La ventaja de la CWT es topográficamente anómala:** los canales más
   discriminativos son frontales (F7, AF3) y temporales (T7, T8), no los
   occipitales y parietales (O1, O2, P7, P8) donde el ritmo alfa debería
   predominar. Esto sugiere que la CWT es más sensible a artefactos de
   origen ocular y muscular que la STFT.

3. **La entropía espectral es degenerada en CWT** (CV = 1.5%, varianza casi
   nula), por lo que no aportará poder discriminativo en esa técnica.

4. **El centro de gravedad (`cog`)** es la feature más robusta y
   fisiológicamente coherente en ambas técnicas.

---

## Decisiones de diseño

### Clasificadores seleccionados

Se usan tres clasificadores complementarios, en lugar de uno solo, para que
los resultados sean robustos ante la elección del modelo:

| Clasificador | Justificación |
|---|---|
| Regresión Logística | Baseline lineal. Si los otros lo superan significativamente, el problema tiene estructura no lineal |
| SVM (kernel RBF) | Robusto con muestras pequeñas y alta dimensión. Maneja relaciones no lineales mediante el parámetro C |
| Random Forest | Maneja multicolinealidad entre canales naturalmente. Entrega importancia de features — permite validar si O1/O2 dominan o no |

### Split de datos: temporal, no aleatorio

Las 236 ventanas tienen 75% de solapamiento — ventanas consecutivas
comparten 1.5 s de señal y son casi idénticas. Un split aleatorio
pondría muestras prácticamente duplicadas en train y test a la vez,
produciendo data leakage e inflando artificialmente el accuracy.

La solución es un **split temporal por bloques**:

> Ventanas   1 – 188  (~80% del registro) → entrenamiento  
> Ventanas 189 – 236  (~20% del registro) → test

Este esquema garantiza que el modelo se evalúa sobre señal que no vio
durante el entrenamiento, respetando la estructura temporal de los datos.

### Escalado

Se aplica `StandardScaler` por separado a STFT y CWT:
- Ajustado **solo sobre el conjunto de entrenamiento** (para no filtrar
  información del test al train).
- Aplicado luego al test.
- Las escalas de STFT y CWT no se mezclan.

### Dos configuraciones de features

Dada la topografía anómala detectada en el notebook 05, se evalúan dos
configuraciones:

**Config A — Todos los canales (56 features):** permite al clasificador
usar cualquier canal, incluyendo los frontales y temporales más
discriminativos según el análisis estadístico.

**Config B — Solo canales relevantes para el alfa (16 features):** restringe
el análisis a O1, O2, P7 y P8 — los cuatro electrodos occipitales y
parietales donde el ritmo alfa debería predominar fisiológicamente.

La comparación entre ambas configuraciones responde una pregunta
metodológica clave: **¿la ventaja de la CWT se mantiene cuando se
restringe a los canales fisiológicamente relevantes, o desaparece porque
dependía de los artefactos en canales frontales y temporales?**

### Métricas de evaluación

Con clases ligeramente desbalanceadas (128 ojos abiertos vs 108 ojos
cerrados) se reportan:
- **Accuracy:** proporción de predicciones correctas.
- **F1-score macro:** promedio del F1 de cada clase, trata ambas clases
  por igual independientemente de su frecuencia.
- **ROC-AUC:** área bajo la curva ROC, mide la capacidad de separación
  independientemente del umbral de decisión.
- **Matriz de confusión:** muestra errores específicos por clase.

---

## Estructura del notebook

**Sección 1:** preparación de datos — split temporal y escalado.

**Sección 2:** entrenamiento y evaluación — 3 clasificadores × 2 técnicas
× 2 configuraciones de features = 12 combinaciones evaluadas. Tabla de
resultados comparativa.

**Sección 3:** importancia de features — análisis del Random Forest para
determinar qué canales dominan realmente la clasificación y si coinciden
con la topografía esperada del ritmo alfa.

**Sección 4:** síntesis y conclusiones — ¿la CWT produce mayor accuracy
que la STFT? ¿Es consistente entre clasificadores y configuraciones?
¿Qué explica los resultados?

In [ ]:
# Cargar dataset tabluar